# Lib


In [16]:
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

from gensim.models import Word2Vec
from node2vec import Node2Vec
from sklearn.decomposition import PCA

import warnings

warnings.filterwarnings("ignore")

# Data


In [17]:
# Load Labels
df_target = pd.read_csv("../../data/github_social_network/musae_git_target.csv")
df_target.columns = df_target.columns.str.strip().str.lower().str.replace(" ", "_")

# Load Edges
df_edges = pd.read_csv("../../data/github_social_network/musae_git_edges.csv")
df_edges.columns = df_edges.columns.str.strip().str.lower().str.replace(" ", "_")

# Load Features
with open("../../data/github_social_network/musae_git_features.json", "r") as f:
    features_json = json.load(f)

In [ ]:
graph = nx.from_pandas_edgelist(df_edges, source="id_1", target="id_2", create_using=nx.Graph())
graph.add_nodes_from(df_target["id"])

In [19]:
node_df = df_target.copy()
edge_df = df_edges.copy()
feature_map = features_json

print(f"nodes: {node_df.shape[0]:,}")
print(f"edges: {edge_df.shape[0]:,}")
print(f"feature map entries: {len(feature_map):,}")
print(f"graph nodes: {graph.number_of_nodes():,}")
print(f"graph edges: {graph.number_of_edges():,}")

nodes: 37,700
edges: 289,003
feature map entries: 37,700
graph nodes: 37,700
graph edges: 289,003


# Feature extraction


In [ ]:
def _feature_lookup(feature_map, node_id: int) -> list[int]:
    return feature_map.get(node_id, feature_map.get(str(node_id), []))


def build_enriched_node_frame(
    node_df: pd.DataFrame,
    feature_map,
    graph: nx.Graph,
) -> pd.DataFrame:
    """Add compact, node-level feature columns onto the original data."""
    enriched_df = node_df.copy()

    degree_map = dict(graph.degree())
    clustering_map = nx.clustering(graph)
    pagerank_map = nx.pagerank(graph, alpha=0.85)

    enriched_df["feature_count"] = enriched_df["id"].map(
        lambda node_id: len(_feature_lookup(feature_map, int(node_id)))
    )
    enriched_df["unique_feature_count"] = enriched_df["id"].map(
        lambda node_id: len(set(_feature_lookup(feature_map, int(node_id))))
    )
    enriched_df["has_feature_vector"] = enriched_df["feature_count"] > 0
    enriched_df["degree"] = enriched_df["id"].map(degree_map).fillna(0).astype(int)
    enriched_df["clustering"] = enriched_df["id"].map(clustering_map).fillna(0.0)
    enriched_df["pagerank"] = enriched_df["id"].map(pagerank_map).fillna(0.0)
    enriched_df["degree_centrality"] = (
        enriched_df["id"].map(nx.degree_centrality(graph)).fillna(0.0)
    )

    return enriched_df

In [ ]:
enriched_node_df = build_enriched_node_frame(node_df, feature_map, graph)
enriched_node_df.head()

# Graph embedding


In [ ]:
def build_embedding_frame(
    graph: nx.Graph,
    *,
    dimensions: int,
    walk_length: int,
    num_walks: int,
    window: int,
    p: float,
    q: float,
    seed: int,
) -> pd.DataFrame:
    walker = Node2Vec(
        graph,
        dimensions=dimensions,
        walk_length=walk_length,
        num_walks=num_walks,
        workers=1,
        p=p,
        q=q,
        seed=seed,
        quiet=True,
    )
    model = walker.fit(window=window, min_count=1, batch_words=64, seed=seed)

    node_order = list(graph.nodes())
    vectors = np.vstack([model.wv[str(node_id)] for node_id in node_order])
    embedding_df = pd.DataFrame(
        vectors, columns=[f"emb_{index:03d}" for index in range(dimensions)]
    )
    embedding_df.insert(0, "id", node_order)
    return embedding_df


def export_embedding_frame(embedding_df: pd.DataFrame, name: str):
    output_path = PROCESSED_DIR / f"{name}.csv"
    embedding_df.to_csv(output_path, index=False)
    return output_path

In [ ]:
deepwalk_embedding_df = build_embedding_frame(
    graph,
    dimensions=64,
    walk_length=30,
    num_walks=10,
    window=10,
    p=1.0,
    q=1.0,
    seed=42,
)

node2vec_embedding_df = build_embedding_frame(
    graph,
    dimensions=64,
    walk_length=30,
    num_walks=10,
    window=10,
    p=0.5,
    q=2.0,
    seed=42,
)

In [ ]:
print(deepwalk_embedding_df.head())

In [ ]:
print(node2vec_embedding_df.head())

# Visualize


In [ ]:
def project_embedding(embedding_df: pd.DataFrame) -> pd.DataFrame:
    feature_columns = [column for column in embedding_df.columns if column != "id"]
    coordinates = PCA(n_components=2, random_state=42).fit_transform(
        embedding_df[feature_columns]
    )
    projection_df = pd.DataFrame(coordinates, columns=["x", "y"])
    projection_df.insert(0, "id", embedding_df["id"].to_numpy())
    return projection_df


def plot_embedding(
    projection_df: pd.DataFrame, labels_df: pd.DataFrame, title: str
) -> None:
    merged_df = projection_df.merge(labels_df[["id", "ml_target"]], on="id", how="left")

    plt.figure(figsize=(10, 8))
    scatter = plt.scatter(
        merged_df["x"],
        merged_df["y"],
        c=merged_df["ml_target"],
        cmap="coolwarm",
        s=14,
        alpha=0.7,
        linewidths=0,
    )
    plt.title(title)
    plt.xlabel("Component 1")
    plt.ylabel("Component 2")
    plt.colorbar(scatter, label="ml_target")
    plt.tight_layout()
    plt.show()

In [ ]:
deepwalk_projection_df = project_embedding(deepwalk_embedding_df)
node2vec_projection_df = project_embedding(node2vec_embedding_df)

plot_embedding(deepwalk_projection_df, node_df, "DeepWalk embedding")
plot_embedding(node2vec_projection_df, node_df, "Node2Vec embedding")

# Export


In [ ]:
deepwalk_output_path = export_embedding_frame(deepwalk_embedding_df, "deepwalk")
node2vec_output_path = export_embedding_frame(node2vec_embedding_df, "node2vec")
enriched_node_df.to_csv("../.../data/processed/node_features_enriched.csv", index=False)